In [ ]:
#%pip install llama-index

In [ ]:
from llama_index.core import Document, VectorStoreIndex

In [ ]:
import openai
from google.colab import userdata
openai.api_key = userdata.get('OPEN_API_KEY')

In [ ]:
##
# 1. 문서를 준비한다
document = [
    Document(text="대한민국의 수도는 서울입니다."),
    Document(text="프랑스의 수도는 파리입니다."),
]

# 2. 인덱스 생성(자동으로 벡터화)
# 각 청크를 openai로 벡터화
# 인메모리방식으로 벡터 스토어에 저장
index = VectorStoreIndex.from_documents(document)

# 3. 쿼리 엔진 생성
query_engine = index.as_query_engine(similarity_top_k=3)

# 4. 쿼리 실행
# 생성한 쿼리엔진에서 쿼리실행
response = query_engine.query("대한민국의 수도는 어디입니까?")
print(response)

서울


In [ ]:
# 2. 인덱스 생성(자동으로 벡터화)
# 인덱스 생성이 어떻게 자동으로 벡터화가 되는지
# VectorStoreIndex.from_documents(document)
# 이 메서드는 내부적으로 두 가지 일을 자동으로 합니다:

  # 1> 텍스트 임베딩 생성
  # 각 Document.text를 모델(예: OpenAI의 text-embedding-3-small 등)을 이용해 고정 길이 벡터로 변환합니다.
  # 예: "대한민국의 수도는 서울입니다." → [0.12, -0.34, ..., 0.56] (수백~수천 차원)

  # 2> 벡터 저장 및 인덱싱
  # 생성된 벡터를 효율적으로 검색할 수 있는 자료 구조(예: FAISS, Annoy 등)에 넣습니다.
  # 이렇게 하면 나중에 유사도 검색(query)이 빠르게 수행됩니다.

  # 즉, from_documents 안에서 임베딩 모델 호출 + 벡터 저장이 자동으로 이루어집니다.

In [ ]:
# 청크 : 문서 검색의 최소단위 모델이 한번에 처리할 수 있는 길이로 잘라낸 텍스트
# 모델 입력길이 제한, 문서가 길면 한번에 처리할 수 없어서 청크로 나눠 처리
# 벡터 ^ : 데이터를 숫자로 바꾸고 방향성을 가지게 한 것
# 벡터DB에서 문서 전체가 아니라 청크단위로 벡터화
# 질문과 유사한 작은 단위를 찾아 답변을 생성
# 전체 문서를 이해하는 대신 청크별로 처리해서 중요한 부분에 집중 - Attention

# 작을수록 : 정확한 검색, 많은 api 호출
# 클수록 : 넓은 컨택스트, 적은 api 호출

In [ ]:
from llama_index.core import Settings

# 청크 크기 설정
Settings.chunk_size = 512 # 기본값
# Setting.chunk_overlap = 128 # 기본값
Settings.chunk_overlap=50 # 청크간 겹침

# 유사도 임계값 설정
from llama_index.core.postprocessor import SimilarityPostprocessor
query_engine = index.as_query_engine(
    similarity_top_k=2, # 유사도 상위 2개
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.7) # 유사도 0.7미만의 문서는 제외(노이즈 제거)
    ]
)
# 배치 처리
Settings.embed_batch_size = 100

### 한국어 데이터로 RAG 구현

In [ ]:

# 라이브러리
from llama_index.core import Document,VectorStoreIndex,Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

In [ ]:
# 1. LLM. 임베딩 모델 설정
Settings.llm = OpenAI(model='gpt-4o-mini',temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
# 2. 문서 준비
documents = [
    Document(
        text="김치는 한국의 대표적인 발효 음식입니다. 배추에 고춧가루, 마늘, 생강 등을 넣어 만듭니다.",
        metadata={"source": "한국 음식 백과", "category": "반찬"}
    ),
    Document(
        text="비빔밥은 밥 위에 여러 가지 나물과 고기, 계란을 올려 고추장과 섞어 먹는 음식입니다.",
        metadata={"source": "한국 음식 백과", "category": "밥 요리"}
    ),
    Document(
        text="불고기는 양념한 소고기를 구워 먹는 한국의 전통 음식입니다. 달콤하고 짭짤한 맛이 특징입니다.",
        metadata={"source": "한국 음식 백과", "category": "고기 요리"}
    ),
    Document(
        text="떡볶이는 가래떡에 고추장 양념을 넣어 볶은 한국의 길거리 음식입니다. 달콤하고 매운 맛이 특징입니다.",
        metadata={"source": "한국 음식 백과", "category": "분식"}
    ),
]

In [ ]:
# 3. 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

In [ ]:
# 4. 쿼리 엔진 생성
index.as_query_engine(
    similarity_top_k=2,
    # node_postprocessors=[SimialrityPostprocessor(similarity_cut_off=0.7)]
)

In [ ]:
# 5. 질문하기
questions = [
    '김치는 어떤 음식인가요?',
    '비빔밥을 어떻게 먹나요?',
    '한국의 고기요리에는 뭐가 있나요?'
]

for q in questions:
  response = query_engine.query(q)
  print(f'질문:{q}, 답변:{response}')

질문:김치는 어떤 음식인가요?, 답변:김치는 한국의 전통 발효 음식입니다.
질문:비빔밥을 어떻게 먹나요?, 답변:비빔밥을 잘 섞어서 고명과 함께 먹습니다.
질문:한국의 고기요리에는 뭐가 있나요?, 답변:한국의 고기요리에는 불고기, 갈비, 삼겹살 등이 있습니다.


#### LLM 캐시

In [ ]:
# 동일한 질문을 반복하면
# (이전에 사용했던 대답)캐시를 (재)사용
import time
start = time.time()
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPEN_API_KEY"))
question = '대한민국의 수도는'
response = client.chat.completions.create(
    model = 'gpt-3.5-turbo',
    messages = [{'role':'user','content':question}],
    temperature=0,
    )
answer = response.choices[0].message.content
elapsed_time = time.time() - start
print(f'elapsed_time : {elapsed_time}')

elapsed_time : 0.8663201332092285


In [ ]:
# 캐시
# 1) 완전 일치 캐시
# 캐시를 딕셔너리로 저장하여 완전 동일한 질문에 해당 캐시를 재사용
start = time.time()
cache = {}
response = []
for i in range(100):
  if question in cache:
    answer = cache[question]
  else :
    response = client.chat.completions.create(
    model = 'gpt-3.5-turbo',
    messages = [{'role':'user','content':question}],
    temperature=0,
    )
    answer = response.choices[0].message.content
    cache[question] = answer
elapsed_time = time.time() - start
print(f'elapsed_time : {elapsed_time}')

elapsed_time : 0.25891780853271484


In [ ]:
# 캐시.. 완전 일치 캐시(Exact Match Cache)
# 동일한 입력 -> 저장된 응답 변환
# 장점 : 구현이 간단하고, 100% 정확
# 단점 : 완전히 같아야만 작동
# 대한민국의 수도는?  캐시 히트
# 대한민국 수도는?    캐시 미스(다른 문자열)
# 한국의 수도는?      캐시 미스

In [ ]:
# 2) 의미적 캐시 (Semantic Cache)
# 의미가 비슷한 입력 -> 저장된 응답 반환
# 1. 유사한 프롬프트 검색
# 2. 유사도 확인 (특정 임계값을 지정해서 그 값에 따라서 답변 채택 종료)
# 3. 비슷한게 없으면 LLM 호출

# 장점 :
# 높은 히트율
# 다양한 표현을 허용 유연함
# 비용 절감

# 단점 :
# 약간 느림
# 벡터 DB 필요

In [ ]:
#%pip install chromadb

In [ ]:
# 벡터 스토어 -> DB : 유사도 기반 검색 기능 지원
# 문서나 텍스트를 벡터로 변환한 후에 저장 -> 유사도 기반 검색 기능
import chromadb
# 클라이언트 생성
client = chromadb.Client()
# 컬렉션 생성
collection = client.create_collection('my_collection_3') # 초기화 명령어 ^

# 문서화 임베딩 준비
texts = [
    '대한민국의 수도는 서울입니다.',
    '프랑스의 수도는 파리 입니다.',
    '서울은 한국의 정치,경제 중심지 입니다.'
]

In [ ]:
from sentence_transformers import SentenceTransformer
# 임베딩 도구 생성
model = SentenceTransformer('all-MiniLM-L6-v2')
# texts 임베딩
embeddings = model.encode(texts).tolist()
embeddings # (3,384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[[-0.023814871907234192,
  0.08707409352064133,
  0.07979419827461243,
  -0.01038955245167017,
  -0.038701750338077545,
  -0.029340991750359535,
  0.1098581999540329,
  0.015440688468515873,
  -0.011003025807440281,
  -0.009104211814701557,
  0.08716967701911926,
  -0.04601135477423668,
  0.049076393246650696,
  -0.07741812616586685,
  0.08031688630580902,
  -0.036522675305604935,
  0.04435153678059578,
  0.05164165049791336,
  -0.10573047399520874,
  0.031718261539936066,
  0.02549372985959053,
  -0.004908771254122257,
  0.029756665229797363,
  0.028301816433668137,
  -0.06878554821014404,
  -0.027313821017742157,
  0.005474111530929804,
  -0.01201446633785963,
  0.07081883400678635,
  0.041178226470947266,
  -0.020179130136966705,
  0.06898639351129532,
  0.02843795157968998,
  0.07568584382534027,
  -0.06454993784427643,
  0.044827885925769806,
  -0.06224128603935242,
  0.019267573952674866,
  -0.017469609156250954,
  -0.008292268961668015,
  -0.15329045057296753,
  -0.1490267664194

In [ ]:
# 문서 추가
ids = ['doc1','doc2', 'doc3']
collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids
)

In [ ]:
# 유사도 검색
query = '한국의 수도는 어디인가요?'
query_embedding = model.encode(query).tolist()
results = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)
print(results)

{'ids': [['doc1']], 'embeddings': None, 'documents': [['대한민국의 수도는 서울입니다.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.39439284801483154]]}


In [ ]:
# 다중 캐시 전략
# 메모리(완전일치) - 벡터DB(의미적) - 미스 LLM호출

In [ ]:
from logging import setLogRecordFactory
# 완전일치 사례의 경우 클래스 생성하기
class SimpleCache:
  def __init__(self):
    self.cache = {} # 딕셔너리
    self.hits = 0
    self.misses = 0
  def get(self,key):
    if key in self.cache:
      self.hits += 1
      return self.cache[key]
    else :
      self.misses += 1
      return None
  def set(self,key,value):
    self.cache[key] = value
  def state(self):
    total = self.hits + self.misses
    hit_rate = self.hits / total*100 if total > 0 else 0
    return {
        'hits' :self.hits,
        'misses' : self.misses,
        'total' : total,
        'hit_rate' : f'{hit_rate:.2f}%'
    }

In [ ]:
from google.colab import userdata
import openai
from openai import OpenAI
# openai.api_key = userdata.get('OPEN_API_KEY')
client = OpenAI(api_key = userdata.get('OPEN_API_KEY'))
def call_llm(question):
  response = client.chat.completions.create(
      model = 'gpt-3.5-turbo',
      messages = [{'role':'user', 'content':question}],
      temperature = 0
    )
  return response.choices[0].message.content

In [ ]:
cache = SimpleCache()
questions=[
    '대한민국의 수도는?',
    '대한민국 수도는?', # 캐시 히트
    '한국의 수도는?'    #  캐시 미스(다른 문자열)
]
for q in questions:
  cached = cache.get(q)
  if cached:
    print(f'캐시 : {cached}')
  else :
    response = call_llm(q)
    cache.set(q,response)
    print(f'llm : resposne')
  print(cache.state)

llm : resposne
<bound method SimpleCache.state of <__main__.SimpleCache object at 0x7ce547715be0>>
llm : resposne
<bound method SimpleCache.state of <__main__.SimpleCache object at 0x7ce547715be0>>
llm : resposne
<bound method SimpleCache.state of <__main__.SimpleCache object at 0x7ce547715be0>>


In [ ]:
##
# 2. 의미적 유사성 - 벡터 DB chromab
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
class SemantiCache:
  def __init__(self,name = 'seamantic_cache'):
    self.client = chromadb.Client()
    self.embed_fn = OpenAIEmbeddingFunction(
        api_key=userdata.get('OPEN_API_KEY'),
        model_name='text-embedding-3-small'
    )
    self.collection = self.client.create_collection(
        name = name,
        embedding_function = self.embed_fn,
        metadata = {'hnsw:space':'cosine'}
    )
  def get(self,query,threshold=0.15):
    results = self.collection.query(
        query_texts=[query],
        n_results =1
    )
    if results['distances'][0] and results['distances'][0][0] < threshold:
      return results['metadatas'][0][0]['response']
    return None
  def set(self,query,response):
    import uuid
    self.collection.add(
        documents=[query],
        metadatas=[{'response':response}],
        ids=[str(uuid.uuid4())]
    )

In [ ]:
import uuid
uuid.uuid4()

UUID('d3e21784-14ad-4f30-8a45-49e8fee0af0c')

In [ ]:
# SemanticCache 사용
cache = SemantiCache(name='test2') # 의미적 유사성을 조회하고 캐시를 매칭하는 클래스

questions=[
    '대한민국의 수도는?',
    '대한민국 수도는?', # 캐시 히트
    '한국의 수도는?'    #  캐시 미스(다른 문자열)
]

for q in questions:
  cached = cache.get(q)
  if cached:
    print(f'HIT :{q} - {cached}')
  else :
    response = call_llm(q)
    cache.set(q,response)
    print(f'MISS : {q} - {response}')

MISS : 대한민국의 수도는? - 서울입니다.
HIT :대한민국 수도는? - 서울입니다.
MISS : 한국의 수도는? - 서울입니다.


### 멀티 캐시

In [ ]:
'''
L1 메모리 내부메모리.. dictionary
L2 메모리 벡터DB - 의미적 유사성
L3 메모리 LLM호출
'''

# 내부메모리
# 임베딩 벡터
# 위를 조회후 없는 경우 LLM 호출
# 총 세개를 동시에 검색 기능 사용

class MultiLevelCache:
  def __init__(self) -> None:
    self.l1_cach = SimpleCache() # 메모리방식 dictionary 완전일치
    self.l2_cach = SemantiCache() # ChromaDB 벡터DB 유사도 방식
  def stats(self):
    print(f'L1 cach :{self.l1_cach.cache}')
  def get(self,key):
    cached = self.l1_cach.get(key)
    if cached:
      print('L1 cache')
      return cached
    cached = self.l2_cach.get(key)
    if cached:
      print('L2 cache')
      self.l1_cach.set(key,cached)
      return cached
    # LLM 호출
    print('LLM')
    response = call_llm(key)
    self.l1_cach.set(key,response)
    self.l2_cach.set(key,response)
    return response
multilevelcache = MultiLevelCache()

In [ ]:
multilevelcache.get('대한민국의 수도는')

LLM


'서울입니다.'

In [ ]:
multilevelcache.stats()

L1 cach :{'대한민국의 수도는': '서울입니다.'}
